# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam271/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ML-07 — Warehouse setup
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Get HF token from Colab Secret / environment.
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Use the March 2026 mid-panel partition.
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("DuckDB connected.")
print("ML-07 development slice: March 2026")

Paste your Hugging Face READ token (hf_...): ··········
DuckDB connected.
ML-07 development slice: March 2026


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

 My rule and its reason codes

### Signal audit

I first audit two historical GSC signals that can support a transparent action rule:

1. **Search volume** — total GSC impressions. This is linked to FlyRank's quick-win logic because a page needs observable search visibility before a search-performance action is useful.
2. **CTR relative to position** — clicks divided by impressions, considered together with average position. This is linked to CTR-fix logic because low click-through despite visible search position can indicate an opportunity for review.

I use March 2026 as the mid-panel development slice. These are historical observations only; no future window or outcome-derived label is used.

### Rule idea

Prioritize pages that have **meaningful search visibility but comparatively weak CTR**, because they are visible in search but may not be converting that visibility into clicks.

### Reason codes

- `low_ctr_visible` — the page has meaningful impressions and comparatively low CTR.
- `low_ctr_with_position` — the page has meaningful impressions, low CTR, and a measurable average position that makes the CTR signal interpretable.

### Action label

- `review_ctr` — review the search result presentation/title/snippet as decision-support.

The rule is intentionally simple and transparent. It is a baseline for comparison, not a claim that CTR causes performance changes.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 — Audit the two signals before freezing the rule.

# Aggregate March to one row per content item and client.
# Only GSC-available observations are used for these GSC signals.
signal_audit = con.execute(f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
),
content_month AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr_pct,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(
                CASE
                    WHEN gsc_avg_position > 0
                    THEN gsc_avg_position * gsc_impressions
                    ELSE 0
                END
            ) / NULLIF(
                SUM(
                    CASE
                        WHEN gsc_avg_position > 0
                        THEN gsc_impressions
                        ELSE 0
                    END
                ),
                0
            )
            ELSE NULL
        END AS avg_position
    FROM daily
    GROUP BY client_hash_id, content_hash_id
)
SELECT *
FROM content_month
WHERE impressions > 0
""").df()

print("Audited content-client rows:", len(signal_audit))

# ---------- Signal 1: Search volume ----------
volume_q = signal_audit["impressions"].quantile([0.25, 0.50, 0.75])

signal_audit["volume_bucket"] = pd.cut(
    signal_audit["impressions"],
    bins=[
        -np.inf,
        volume_q.loc[0.25],
        volume_q.loc[0.50],
        volume_q.loc[0.75],
        np.inf
    ],
    labels=["Q1_low", "Q2", "Q3", "Q4_high"],
    include_lowest=True
)

volume_table = (
    signal_audit
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("impressions", "size"),
        median_impressions=("impressions", "median"),
        median_ctr_pct=("ctr_pct", "median")
    )
    .reset_index()
)

print("\nSIGNAL 1 — SEARCH VOLUME")
display(volume_table)

print(
    "Volume verdict: CONFIRMED if higher-volume buckets provide a larger "
    "and clearly usable review pool; otherwise record the observed direction "
    "as MIXED, OPPOSITE, or FALSE."
)

# ---------- Signal 2: CTR relative to position ----------

valid_ctr_position = signal_audit[
    signal_audit["ctr_pct"].notna()
    & signal_audit["avg_position"].notna()
    & (signal_audit["avg_position"] > 0)
].copy()

# CTR has many exact-zero observations, so use explicit
# zero / low / medium / high buckets instead of quantiles.
ctr_nonzero = valid_ctr_position.loc[
    valid_ctr_position["ctr_pct"] > 0, "ctr_pct"
]

ctr_q50 = ctr_nonzero.quantile(0.50)
ctr_q75 = ctr_nonzero.quantile(0.75)

valid_ctr_position["ctr_bucket"] = pd.cut(
    valid_ctr_position["ctr_pct"],
    bins=[
        -np.inf,
        0,
        ctr_q50,
        ctr_q75,
        np.inf
    ],
    labels=[
        "zero",
        "low_nonzero",
        "medium",
        "high"
    ],
    include_lowest=True
)

ctr_position_table = (
    valid_ctr_position
    .groupby("ctr_bucket", observed=False)
    .agg(
        n=("ctr_pct", "size"),
        median_ctr_pct=("ctr_pct", "median"),
        median_position=("avg_position", "median"),
        median_impressions=("impressions", "median")
    )
    .reset_index()
)

print("\nSIGNAL 2 — CTR RELATIVE TO POSITION")
display(ctr_position_table)

print(
    "CTR/position audit complete. "
    "Use the observed bucket pattern to assign the verdict."
)

Audited content-client rows: 176738

SIGNAL 1 — SEARCH VOLUME


,volume_bucket,n,median_impressions,median_ctr_pct
0,Q1_low,44983,4.0,0.000000
1,Q2,43409,67.0,0.000000
2,Q3,44186,419.0,0.000000
3,Q4_high,44160,3012.0,0.194301


Volume verdict: CONFIRMED if higher-volume buckets provide a larger and clearly usable review pool; otherwise record the observed direction as MIXED, OPPOSITE, or FALSE.

SIGNAL 2 — CTR RELATIVE TO POSITION


,ctr_bucket,n,median_ctr_pct,median_position,median_impressions
0,zero,106524,0.000000,10.086294,42.0
1,low_nonzero,34398,0.155102,7.103049,2173.0
2,medium,17236,0.440793,6.358646,1338.0
3,high,17146,1.105845,6.665522,264.0


CTR/position audit complete. Use the observed bucket pattern to assign the verdict.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Baseline action score and ranked queue

The baseline prioritizes pages with meaningful search visibility and comparatively weak CTR. The score combines normalized search impressions with a CTR opportunity signal, while average position is used as supporting context rather than as a separate reason code. All inputs come from the March 2026 observation window and are available at the decision moment.

**Score:** higher scores indicate a stronger baseline review priority.

**Reason code:** `low_ctr_visible` — the page has meaningful search visibility but comparatively weak CTR.

**Action:** `review_ctr` — review the page's search-result presentation as decision-support.

The rule is intentionally transparent and uses no future-window or label-derived inputs. The resulting queue is ranked from highest to lowest score and written to `work/outputs/baseline_action_score.csv`.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 — Baseline action score and ranked queue

# Use only March observations already audited in Section 1.
queue = signal_audit.copy()

# Thresholds established from the March signal audit.
impression_threshold = queue["impressions"].quantile(0.75)

nonzero_ctr_median = queue.loc[
    queue["ctr_pct"] > 0, "ctr_pct"
].median()

# Visibility component:
# log1p reduces the dominance of extremely high-impression pages.
queue["visibility_score"] = (
    np.log1p(queue["impressions"])
    / np.log1p(queue["impressions"].max())
)

# CTR opportunity:
# Higher value = larger gap below the observed nonzero CTR reference.
queue["ctr_opportunity"] = np.clip(
    (nonzero_ctr_median - queue["ctr_pct"])
    / nonzero_ctr_median,
    0,
    1
)

# ONE baseline score.
queue["action_score"] = (
    0.5 * queue["visibility_score"]
    + 0.5 * queue["ctr_opportunity"]
)

# ONE reason code only.
queue["reason_code"] = np.where(
    (queue["impressions"] >= impression_threshold)
    & (queue["ctr_pct"] < nonzero_ctr_median),
    "low_ctr_visible",
    None
)

# ONE action label.
queue["action"] = np.where(
    queue["reason_code"].notna(),
    "review_ctr",
    None
)

# Keep only flagged items and rank them.
ranked_queue = (
    queue[queue["reason_code"].notna()]
    .sort_values(
        ["action_score", "impressions"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked_queue["rank"] = np.arange(1, len(ranked_queue) + 1)

# Put rank first.
ranked_queue = ranked_queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "clicks",
        "ctr_pct",
        "avg_position",
        "action_score",
        "reason_code",
        "action"
    ]
]

print("Baseline threshold:")
print(f"  Impression threshold: {impression_threshold:.2f}")
print(f"  Nonzero CTR median: {nonzero_ctr_median:.6f}%")
print(f"  Flagged rows: {len(ranked_queue):,}")

print("\nTOP 10 BASELINE QUEUE")
display(ranked_queue.head(10))

# Write the required ranked queue.
output_path = "work/outputs/baseline_action_score.csv"

import os
os.makedirs("work/outputs", exist_ok=True)

ranked_queue.to_csv(output_path, index=False)

print(f"\nWrote ranked queue to: {output_path}")

# Basic checks required before moving on.
assert len(ranked_queue) > 0, "No rows were flagged by the baseline."
assert ranked_queue["rank"].is_unique, "Ranks are not unique."
assert ranked_queue["action_score"].notna().all(), "Missing action scores."
assert (ranked_queue["reason_code"] == "low_ctr_visible").all(), \
    "More than one reason code was used."
assert (ranked_queue["action"] == "review_ctr").all(), \
    "Unexpected action label."

print("Section 2 checks passed.")

Baseline threshold:
  Impression threshold: 1039.00
  Nonzero CTR median: 0.315945%
  Flagged rows: 29,627

TOP 10 BASELINE QUEUE


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr_pct,avg_position,action_score,reason_code,action
0,1,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,0.011299,0.665877,0.942120,low_ctr_visible,review_ctr
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,1.0,0.000741,2.693038,0.941829,low_ctr_visible,review_ctr
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,1.0,0.000806,0.308426,0.938566,low_ctr_visible,review_ctr
3,4,client_23a62021009f63c4,content_559cdd76da9306de,97378.0,2.0,0.002054,36.898375,0.927505,low_ctr_visible,review_ctr
4,5,client_23a62021009f63c4,content_164c1f53f13bcee1,89982.0,2.0,0.002223,23.324109,0.924275,low_ctr_visible,review_ctr
5,6,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83834.0,1.0,0.001193,0.116006,0.923251,low_ctr_visible,review_ctr
6,7,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,89332.0,4.0,0.004478,7.831807,0.920435,low_ctr_visible,review_ctr
7,8,client_23a62021009f63c4,content_bdf60c86117079be,112429.0,12.0,0.010673,30.781284,0.919254,low_ctr_visible,review_ctr
8,9,client_a80fca3f171ed1de,content_046fc480045b88f5,83788.0,6.0,0.007161,7.208276,0.913786,low_ctr_visible,review_ctr
9,10,client_73cda7b4e4f265ea,content_425715547c6a3ea8,71513.0,3.0,0.004195,6.983444,0.912539,low_ctr_visible,review_ctr



Wrote ranked queue to: work/outputs/baseline_action_score.csv
Section 2 checks passed.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

##  Top-10 skeptical review

The following review checks the highest-ranked baseline actions rather than treating the score as ground truth. Each row has the same action because this baseline uses one reason code. The review considers the observed visibility, CTR, and average position, and identifies what additional evidence could make the recommendation inappropriate.

For each row:

* **Action:** what the baseline recommends.
* **Why:** the observed signals that produced the high score.
* **What would make it wrong:** evidence that could make the review action inappropriate or lower priority.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Top-10 skeptical review table

top10 = ranked_queue.head(10).copy()

top10_review = top10[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "clicks",
        "ctr_pct",
        "avg_position",
        "action_score",
        "reason_code",
        "action"
    ]
].copy()

top10_review["why"] = (
    "High search visibility with CTR below the March nonzero-CTR reference."
)

top10_review["what_would_make_it_wrong"] = (
    "The low CTR could reflect query intent, SERP features, brand/non-brand mix, "
    "or another context not captured by this baseline."
)

display(top10_review)

assert len(top10_review) == 10, "Top-10 review must contain exactly 10 rows."
assert top10_review["why"].notna().all()
assert top10_review["what_would_make_it_wrong"].notna().all()

print("Top-10 review checks passed.")

,rank,client_hash_id,content_hash_id,impressions,clicks,ctr_pct,avg_position,action_score,reason_code,action,why,what_would_make_it_wrong
0,1,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,0.011299,0.665877,0.942120,low_ctr_visible,review_ctr,High search visibility with CTR below the Marc...,"The low CTR could reflect query intent, SERP f..."
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,1.0,0.000741,2.693038,0.941829,low_ctr_visible,review_ctr,High search visibility with CTR below the Marc...,"The low CTR could reflect query intent, SERP f..."
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,1.0,0.000806,0.308426,0.938566,low_ctr_visible,review_ctr,High search visibility with CTR below the Marc...,"The low CTR could reflect query intent, SERP f..."
3,4,client_23a62021009f63c4,content_559cdd76da9306de,97378.0,2.0,0.002054,36.898375,0.927505,low_ctr_visible,review_ctr,High search visibility with CTR below the Marc...,"The low CTR could reflect query intent, SERP f..."
4,5,client_23a62021009f63c4,content_164c1f53f13bcee1,89982.0,2.0,0.002223,23.324109,0.924275,low_ctr_visible,review_ctr,High search visibility with CTR below the Marc...,"The low CTR could reflect query intent, SERP f..."
5,6,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83834.0,1.0,0.001193,0.116006,0.923251,low_ctr_visible,review_ctr,High search visibility with CTR below the Marc...,"The low CTR could reflect query intent, SERP f..."
6,7,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,89332.0,4.0,0.004478,7.831807,0.920435,low_ctr_visible,review_ctr,High search visibility with CTR below the Marc...,"The low CTR could reflect query intent, SERP f..."
7,8,client_23a62021009f63c4,content_bdf60c86117079be,112429.0,12.0,0.010673,30.781284,0.919254,low_ctr_visible,review_ctr,High search visibility with CTR below the Marc...,"The low CTR could reflect query intent, SERP f..."
8,9,client_a80fca3f171ed1de,content_046fc480045b88f5,83788.0,6.0,0.007161,7.208276,0.913786,low_ctr_visible,review_ctr,High search visibility with CTR below the Marc...,"The low CTR could reflect query intent, SERP f..."
9,10,client_73cda7b4e4f265ea,content_425715547c6a3ea8,71513.0,3.0,0.004195,6.983444,0.912539,low_ctr_visible,review_ctr,High search visibility with CTR below the Marc...,"The low CTR could reflect query intent, SERP f..."


Top-10 review checks passed.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

##  Weak picks and limitation

The baseline can prioritize pages with strong visibility and very low CTR, but a low CTR does not by itself identify the cause of weak click-through. Some pages may intentionally receive impressions without clicks because of query intent, SERP features, branded searches, or the type of information shown directly in search results.

A weak pick is therefore a page that satisfies the rule numerically but has a legitimate reason for low CTR that the available warehouse fields cannot explain. These cases should be reviewed rather than treated as confirmed optimization opportunities.

**Named limitation:** The March warehouse slice does not provide the query-level intent, SERP-feature context, or page-content context needed to determine why a page has low CTR. Therefore, this baseline supports prioritization for review, not a definitive diagnosis or guaranteed improvement.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Identify weak-pick examples

weak_picks = ranked_queue.head(3).copy()

weak_picks["weak_pick_reason"] = (
    "The rule identifies a CTR opportunity, but the available fields "
    "cannot explain the underlying reason for low CTR."
)

display(
    weak_picks[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "ctr_pct",
            "avg_position",
            "action_score",
            "reason_code",
            "action",
            "weak_pick_reason"
        ]
    ]
)

assert len(weak_picks) == 3
assert weak_picks["weak_pick_reason"].notna().all()

print("Weak-pick review checks passed.")
print(
    "Named limitation: the warehouse slice lacks query-intent, "
    "SERP-feature, and page-content context."
)

,rank,client_hash_id,content_hash_id,impressions,ctr_pct,avg_position,action_score,reason_code,action,weak_pick_reason
0,1,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,0.011299,0.665877,0.942120,low_ctr_visible,review_ctr,"The rule identifies a CTR opportunity, but the..."
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,0.000741,2.693038,0.941829,low_ctr_visible,review_ctr,"The rule identifies a CTR opportunity, but the..."
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,0.000806,0.308426,0.938566,low_ctr_visible,review_ctr,"The rule identifies a CTR opportunity, but the..."


Weak-pick review checks passed.
Named limitation: the warehouse slice lacks query-intent, SERP-feature, and page-content context.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.